In [1]:
#### AI모델이 대화를 처리할 때, 처리해야 할 정보의 양을 줄여서 더 빠르고 효율적으로 응답할 수 있도록 하는 방법

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import trim_messages
from langchain_openai import ChatOpenAI
from operator import itemgetter
from dotenv import load_dotenv

load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")


store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]



prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)


# 메시지 트리밍 유틸리티 설정( 가장 최근 2개의 메세지만 유지 )
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    #RunnablePassthrough.assign()은 반드시 딕셔너리(dict) 입력만 받음
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | model
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    # input_messages_key, history_messages_key 항목을 제공하지 않으면 메시지 리스트(list) 형태로 변환하여 반환됨
)

# 새로운 대화 내용 추가 후 체인 실행
answer = chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content
print(answer)


d:\source\ai\langchain_app\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


5년 내에 집을 사기 위해서는 체계적인 재정 계획이 필요합니다. 다음은 그 과정에서 참고할 수 있는 단계입니다.

### 1. **목표 설정**
   - **예산 설정**: 원하는 집의 가격대를 파악하세요. 지역, 크기, 형태에 따라 가격이 다르므로 미리 조사해보세요.
   - **다운 페이먼트**: 대부분의 경우 주택 구입 시 20%의 다운 페이먼트를 요구합니다. 목표 구매 가격에 따라 얼마를 저축해야 할지 계산해보세요.

### 2. **저축 계획**
   - **저축 목표 설정**: 목표하는 다운 페이먼트를 계산하고, 이를 5년 동안 어떻게 저축할 것인지 계획하세요.
   - **매달 저축 금액**: 
     - 예를 들어, 3억 원의 집을 구매한다고 가정하면 20%인 6천만 원이 필요합니다. 5년(60개월) 동안 저축하려면 매달 100만 원을 저축해야 합니다.
   - **저축 계좌**: 별도의 저축 계좌를 만들어 목표에 도달할 때까지 분리해서 관리하세요. 이자를 고려하면 훨씬 수월할 수 있습니다.

### 3. **재정 상태 점검**
   - **소득 및 지출 분석**: 현재 소득과 지출을 점검해 필요 없는 지출을 줄이고 저축할 수 있는 여력을 높이세요.
   - **부채 관리**: 부채가 있는 경우 이를 줄이기 위해 힘쓰세요. 주택 구매 시 추가적인 대출이 필요할 수 있으므로 신용 점수를 높이는 것이 중요합니다.

### 4. **투자 고려**
   - **투자 옵션**: 단순 저축 외에도 적절한 투자 방법을 고려해 자산을 늘릴 수 있습니다. 위험을 감수할 수 있는 수준이라면, ETF, 주식, 채권 등에 투자해볼 수 있습니다.

### 5. **신용 점검 및 관리**
   - **신용 점수 관리**: 좋은 신용 점수는 대출 이자율에 긍정적인 영향을 미칩니다. 신용 카드 사용을 주의하고, 채무를 제때 갚아 신용 점수를 관리하세요.

### 6. **전문가 상담**
   - **부동산 전문가 혹은 금융 상담사와 상담**: 부동산 시장의 변화, 금융 상

In [ ]:
answer = chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}
).content
print(answer)

당신은 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하는지 질문하셨습니다. 이에 대한 재정 계획의 단계와 조언을 드렸습니다. 추가적인 질문이나 궁금한 점이 있다면 말씀해 주세요!
